In [60]:
def load_data():
    import pandas as pd
    return pd.read_csv("data.csv")  # default dataset


In [61]:
def validate_data(df):
    if df.empty:
        raise ValueError("Dataset is empty")
    return True


In [62]:
def detect_columns(df):
    num_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
    cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
    return num_cols, cat_cols


In [63]:
def detect_target(df):
    common_targets = ["target", "label", "y", "churn", "class"]
    for col in df.columns:
        if col.lower() in common_targets:
            return col
    return None


In [64]:
def fix_numeric_strings(df):
    import pandas as pd

    for col in df.columns:
        if df[col].dtype == "object":
            try:
                df[col] = pd.to_numeric(df[col])
            except:
                pass
    return df


In [65]:
def drop_id_columns(df):
    id_cols = [col for col in df.columns if "id" in col.lower()]
    return df.drop(columns=id_cols), id_cols


In [66]:
def detect_missing(df, threshold=0.2):
    return df.columns[df.isnull().mean() > threshold].tolist()


In [67]:
def detect_outliers(df, num_cols):
    from sklearn.ensemble import IsolationForest

    if not num_cols:
        return df

    iso = IsolationForest(contamination=0.05, random_state=42)
    df["outlier_flag"] = iso.fit_predict(df[num_cols])
    return df


In [68]:
def build_preprocessor(num_cols, cat_cols):
    from sklearn.pipeline import Pipeline
    from sklearn.compose import ColumnTransformer
    from sklearn.preprocessing import StandardScaler, OneHotEncoder
    from sklearn.impute import SimpleImputer

    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="mean")),
        ("scaler", StandardScaler())
    ])

    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    return ColumnTransformer([
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols)
    ])


In [69]:
def transform_data(df, preprocessor, target_col):
    if target_col and target_col in df.columns:
        df = df.drop(columns=[target_col])

    processed = preprocessor.fit_transform(df)
    return processed


In [70]:
def save_processed_data(processed_data, preprocessor, df):
    import pandas as pd

    feature_names = preprocessor.get_feature_names_out()
    df_out = pd.DataFrame(processed_data.toarray(), columns=feature_names)

    df_out.to_csv("processed_data.csv", index=False)


In [71]:
def run_preprocessing():
    df = load_data()

    # Drop ID columns
    df, dropped_ids = drop_id_columns(df)

    # Fix numeric strings like TotalCharges
    df = fix_numeric_strings(df)

    # Detect & drop target
    target_col = detect_target(df)
    if target_col:
        df = df.drop(columns=[target_col])

    # Detect columns AFTER cleaning
    num_cols, cat_cols = detect_columns(df)

    # Outlier detection
    df = detect_outliers(df, num_cols)

    # Preprocess
    preprocessor = build_preprocessor(num_cols, cat_cols)
    processed_data = preprocessor.fit_transform(df)

    # Save with names
    save_processed_data(processed_data, preprocessor, df)

    print("✅ Fully automated preprocessing completed")


In [72]:
run_preprocessing()


✅ Fully automated preprocessing completed
